In [1]:
from urllib.parse import quote_plus
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np
import random

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [4]:
dfp = pd.read_csv('C:/Users/ocordoba/OneDrive - Axo/web_scraping/productos_ws.csv')
dfp.head(2)

,campaign_PK,product_PK,product_name,product_physicalreference,ordetail_created,Articulos_vendidos,buyers,visitas,visitors,precio_provedor,precio_privalia,CR,decil_visitors,decil_CR
0,341170,64161622,NAUTICA VOYAGE N-83 EAU DE TOILETTE 100 ML,938230-UNICO,27/10/2025,6,5,29,2,767.24,275.0012,2.5,1,10
1,341170,64160797,MONTBLANC LEGEND BLUE EAU DE PARFUM 100 ML,1C7L027A01-UNICO,27/10/2025,2,2,13,1,2284.48,1137.9948,2.0,1,10


In [5]:
def build_palacio_search_url(query: str) -> str:
    """
    Construye el URL de búsqueda de Palacio de Hierro
    a partir de una frase de búsqueda.

    Ejemplo:
    >>> build_palacio_search_url("HUGO BOSS ORANGE WOMAN EAU DE TOILETTE 75 ML")
    'https://www.elpalaciodehierro.com/buscar?q=HUGO+BOSS+ORANGE+WOMAN+EAU+DE+TOILETTE+75+ML'
    """
    BASE = "https://www.elpalaciodehierro.com/buscar?q="
    query = " ".join(query.split()).strip()  # limpia espacios dobles
    return f"{BASE}{quote_plus(query)}"

In [6]:
import time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

# --- Solo en Windows: detectar versión instalada de Chrome ---
def _chrome_major_windows() -> int | None:
    try:
        import winreg
        # Ruta típica: HKCU\Software\Google\Chrome\BLBeacon\version
        key = winreg.OpenKey(winreg.HKEY_CURRENT_USER, r"Software\Google\Chrome\BLBeacon")
        version_str, _ = winreg.QueryValueEx(key, "version")
        winreg.CloseKey(key)
        # version_str ej: "141.0.7390.123" -> major=141
        return int(version_str.split(".")[0])
    except Exception:
        return None

def get_palacio_html_dynamic(url: str, wait_seconds: int = 10) -> str:
    """
    Abre la página de resultados en un navegador real y
    devuelve el HTML ya renderizado.
    Ajusta automáticamente version_main del driver a la versión de Chrome instalada (Windows).
    """
    major = _chrome_major_windows()  # p.ej. 141
    if major is None:
        # Si no estás en Windows o no se pudo leer, forzamos 141 (ajústalo si actualizas)
        major = 141

    options = uc.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--lang=es-MX")
    options.add_argument("--disable-blink-features=AutomationControlled")

    # Tips de estabilidad en Windows
    # - use_subprocess ayuda a aislar el proceso
    # - headless=False para evitar bloqueos por render
    driver = uc.Chrome(
        options=options,
        headless=False,
        version_main=major,     # 👈 empata con tu Chrome instalado (141)
        use_subprocess=True
    )
    try:
        driver.get(url)
        time.sleep(wait_seconds)  # espera al JS / red

        # Si hay botón de aceptar cookies, intenta dar clic (no siempre es necesario)
        try:
            # Example (si cambia, lo ignoramos sin romper)
            btns = driver.find_elements(By.CSS_SELECTOR, "button, a")
            for b in btns:
                txt = (b.text or "").strip().lower()
                if "acept" in txt or "entendido" in txt or "continuar" in txt:
                    b.click()
                    time.sleep(2)
                    break
        except Exception:
            pass

        html = driver.page_source or ""
        return html
    finally:
        driver.quit()

In [7]:
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import re

PALACIO_BASE = "https://www.elpalaciodehierro.com"

def _parse_price(text: str) -> float | None:
    """
    Convierte precios tipo '$7,500.00' o '7,500.00 MXN' a float (7500.00).
    Si no encuentra número, devuelve None.
    """
    if not text:
        return None
    # Quita todo excepto dígitos y puntos/comas, luego normaliza
    t = text.strip()
    # Busca un bloque de números con miles y decimales opcionales
    m = re.search(r"(\d{1,3}(?:[.,]\d{3})*(?:[.,]\d{2})|\d+(?:[.,]\d{2})?)", t)
    if not m:
        return None
    num = m.group(1)
    # Normaliza: quita separador de miles y usa punto como decimal
    if num.count(",") > 0 and num.count(".") > 0:
        # Asumimos formato 7,500.00 -> miles con coma, decimales con punto
        num = num.replace(",", "")
    else:
        # Si solo hay comas, podrían ser decimales: 7500,00
        # Heurística: si hay una coma y 2 decimales, cambia a punto
        if "," in num and re.search(r",\d{2}$", num):
            num = num.replace(".", "").replace(",", ".")
        else:
            num = num.replace(",", "")
    try:
        return float(num)
    except ValueError:
        return None

def parse_palacio_search_html(html: str) -> pd.DataFrame:
    """
    Recibe el HTML (renderizado con Selenium) de resultados de búsqueda de Palacio
    y devuelve un DataFrame con columnas: url, descripcion, precio_text, precio.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Contenedor principal (varía poco, pero dejamos 2-3 opciones)
    grid = soup.select_one("section.l-plp-grid_wrapper-products") \
           or soup.select_one("div.l-plp-grid_wrapper-products") \
           or soup

    # Cada tarjeta de producto
    items = grid.select("article.m-product, article.l-plp-grid_item, div.m-product, li.m-product")
    rows = []

    for card in items:
        # --- LINK + DESCRIPCION ---
        # Selectores típicos en SFCC/Demandware
        a = (card.select_one("h3.b-product_tile-name a")
             or card.select_one("a.b-product_link")
             or card.select_one("a[data-js-product-tile-name]")
             or card.select_one("a[href*='.html']"))

        url = None
        desc = None
        if a:
            href = a.get("href", "").strip()
            url = urljoin(PALACIO_BASE, href) if href else None
            # texto limpio
            desc = (a.get_text(" ", strip=True) or None)

        # --- PRECIO ---
        # Muchas veces hay dos: precio lista y descuento. Tomamos el “sales” o value principal.
        price_node = (card.select_one("span.b-product_price-value")
                      or card.select_one("[data-js-line-item-price-sales] .b-product_price-value")
                      or card.select_one("[data-js-line-item-price-sales]")
                      or card.select_one(".b-product_price .b-product_price-value"))
        precio_text = None
        precio_val = None
        if price_node:
            # Algunos sitios ponen <span content="7500.00"> $7,500.00 </span>
            precio_text = price_node.get_text(" ", strip=True)
            if not precio_text:
                precio_text = price_node.get("content", None)
            # Si hay atributo content, úsalo como “verdad”
            precio_val = _parse_price(price_node.get("content")) if price_node.get("content") else _parse_price(precio_text)

        # Filtra solo tarjetas con URL y descripción
        if url and desc:
            rows.append({
                "url": url,
                "descripcion": desc,
                "precio_text": precio_text,
                "precio": precio_val
            })

    return pd.DataFrame(rows)


# 🔹 Ejemplo de uso con tu html renderizado (de Selenium Paso 2b):
# html = get_palacio_html_dynamic(url, wait_seconds=10)
# df = parse_palacio_search_html(html)
# print(df.head())
# df.to_csv("palacio_resultados.csv", index=False, encoding="utf-8")


In [8]:
i = 15
query =  list(dfp.product_name)[i]
print(query)
url = build_palacio_search_url(query)
html = get_palacio_html_dynamic(url, wait_seconds=10)
df_scrap = parse_palacio_search_html(html)
print(df_scrap.head())

DAVIDOFF COOL WATER EAU DE TOILETTE 100 ML
                                                 url  \
0  https://www.elpalaciodehierro.com/penhaligons-...   
1  https://www.elpalaciodehierro.com/penhaligons-...   
2  https://www.elpalaciodehierro.com/murad-gel-hi...   
3  https://www.elpalaciodehierro.com/isdin-protec...   
4  https://www.elpalaciodehierro.com/isdin-protec...   

                                         descripcion precio_text  precio  
0  Perfume A Balm of Calm Eau de Parfum, 100 ml U...   $6,500.00  6500.0  
1  Perfume Vra Vra VRoom Eau de Parfum, 100 ml Un...   $6,500.00  6500.0  
2       Gel Hidratante Nutrient Charged Water, 50 ml   $1,970.00  1970.0  
3     Protector Solar Fusion Water Magic Glow, 50 ml     $789.00   789.0  
4     Protector Solar Age Repair Fusion Water, 50 ml     $847.00   847.0  


In [9]:
print(df_scrap.head())

                                                 url  \
0  https://www.elpalaciodehierro.com/penhaligons-...   
1  https://www.elpalaciodehierro.com/penhaligons-...   
2  https://www.elpalaciodehierro.com/murad-gel-hi...   
3  https://www.elpalaciodehierro.com/isdin-protec...   
4  https://www.elpalaciodehierro.com/isdin-protec...   

                                         descripcion precio_text  precio  
0  Perfume A Balm of Calm Eau de Parfum, 100 ml U...   $6,500.00  6500.0  
1  Perfume Vra Vra VRoom Eau de Parfum, 100 ml Un...   $6,500.00  6500.0  
2       Gel Hidratante Nutrient Charged Water, 50 ml   $1,970.00  1970.0  
3     Protector Solar Fusion Water Magic Glow, 50 ml     $789.00   789.0  
4     Protector Solar Age Repair Fusion Water, 50 ml     $847.00   847.0  


In [10]:
columnas = ["pagina","product_PK", "product_name", "campaign_PK", "precio_privalia", "n_productos","precio_min", "precio_max", "precio_avg", "link_min"]
df = pd.DataFrame(columns=columnas)

In [11]:
inicio = time.time()
for i in range(2):
    print(f" producto {i}")
    query =  list(dfp.product_name)[i]
    print(f" buscando el producto: {query}. Con un precio privalio de {list(dfp.precio_privalia)[i]}")
    url = build_palacio_search_url(query)
    html = get_palacio_html_dynamic(url, wait_seconds=10)
    df_scrap = parse_palacio_search_html(html)
    print(df.head())
    #df.to_csv("palacio_resultados.csv", index=False, encoding="utf-8")
    #query_emb = model.encode(query, convert_to_tensor=True)
    print (f'se escrapearon {df_scrap.shape[0]} articulos')
    print(query)
    df_scrap["titulo"] = df_scrap["descripcion"].str.split(",").str[0]
    df_scrap["embedding"] = df_scrap["titulo"].apply(lambda x: model.encode(str(x), convert_to_tensor=True))
    query_emb = model.encode(query, convert_to_tensor=True)
    df_scrap["similaridad"] = df_scrap["embedding"].apply(lambda emb: util.cos_sim(query_emb, emb).item())
    p_min =  list(dfp.precio_privalia)[i]*0.8
    p_max = list(dfp.precio_privalia)[i]*1.2
    #df_sal = df_scrap[(df_scrap.precio> p_min) & (df_scrap.precio< p_max) &(df_scrap.similaridad>=0.7)]
    df_sal = df_scrap[(df_scrap.precio> p_min)]
    print(f'columnas: {df_sal.columns}')
    n_productos = df_sal.shape[0]
    print(f'quedan {n_productos} productos filtrados')
    min_p = df_sal.precio.min()
    print(f'el precio mínimo es: {min_p}')
    max_p = df_sal.precio.max()
    avg = df_sal.precio.mean()
    if n_productos != 0:
            #link_page1 = df_sal[df_sal.precio_actual == min_p].link
            link_min = list(df_sal[df_sal.precio == min_p].url)[0]
    
    else: 
        link_min = np.nan
    df.loc[len(df)] = ["PH", list(dfp.product_PK)[i], list(dfp.product_name)[i], list(dfp.campaign_PK)[i], list(dfp.precio_privalia)[i],df_sal.shape[0], min_p, max_p, avg,link_min ]
    espera=random.normalvariate(30, 10)
    print(f'segundos de sespera {espera}')
    time.sleep(abs(espera))
fin = time.time()
duracion = fin - inicio
print(f"Duración: {fin - inicio:.4f} segundos")

 producto 0
 buscando el producto: NAUTICA VOYAGE N-83 EAU DE TOILETTE 100 ML. Con un precio privalio de 275.0012
Empty DataFrame
Columns: [pagina, product_PK, product_name, campaign_PK, precio_privalia, n_productos, precio_min, precio_max, precio_avg, link_min]
Index: []
se escrapearon 20 articulos
NAUTICA VOYAGE N-83 EAU DE TOILETTE 100 ML
columnas: Index(['url', 'descripcion', 'precio_text', 'precio', 'titulo', 'embedding',
       'similaridad'],
      dtype='object')
quedan 20 productos filtrados
el precio mínimo es: 2500.0
segundos de sespera 25.510839163941483
 producto 1
 buscando el producto: MONTBLANC LEGEND BLUE  EAU DE PARFUM 100 ML. Con un precio privalio de 1137.9948
  pagina  product_PK                                product_name  campaign_PK  \
0     PH    64161622  NAUTICA VOYAGE N-83 EAU DE TOILETTE 100 ML       341170   

   precio_privalia  n_productos  precio_min  precio_max  precio_avg  \
0         275.0012           20      2500.0     12130.0      6794.5   

     